In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from PIL import Image
import warnings
warnings.filterwarnings('ignore')

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("✅ Libraries imported successfully!")

In [ ]:
data_dir = Path("../data/raw")
categories = ["cardboard", "glass", "metal", "paper", "plastic", "trash"]

# Get all image paths
image_data = []

for category in categories:
    cat_path = data_dir / category
    if cat_path.exists():
        for img_path in cat_path.glob("*.jpg"):
            image_data.append({
                'path': str(img_path),
                'category': category,
                'filename': img_path.name
            })
df = pd.DataFrame(image_data)
print(f"📊 Total images found: {len(df)}")
print(f"📂 Categories: {df['category'].nunique()}")
df.head()

In [ ]:
plt.figure(figsize=(10, 6))
category_counts = df['category'].value_counts()
sns.barplot(x=category_counts.index, y=category_counts.values)
plt.title('Distribution of Waste Categories', fontsize=16, fontweight='bold')
plt.xlabel('Category', fontsize=12)
plt.ylabel('Number of Images', fontsize=12)
plt.xticks(rotation=45)

# Add count labels on bars
for i, v in enumerate(category_counts.values):
    plt.text(i, v + 10, str(v), ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('../results/plots/class_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n📈 Class Distribution:")
print(category_counts)
print(f"\n⚖️  Balance ratio (min/max): {category_counts.min() / category_counts.max():.2f}")


In [ ]:
fig, axes = plt.subplots(3, 6, figsize=(18, 10))
fig.suptitle('Sample Images from Each Category', fontsize=16, fontweight='bold')

for idx, category in enumerate(categories):
    # Get 3 random samples from each category
    samples = df[df['category'] == category].sample(3)
    
    for i, (_, row) in enumerate(samples.iterrows()):
        ax = axes[i, idx]
        img = Image.open(row['path'])
        ax.imshow(img)
        ax.axis('off')
        if i == 0:
            ax.set_title(category.capitalize(), fontweight='bold')

plt.tight_layout()
plt.savefig('../results/plots/sample_images.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
print("📐 Analyzing image dimensions...")

dimensions = []
for _, row in df.sample(min(500, len(df))).iterrows():  # Sample for speed
    try:
        img = Image.open(row['path'])
        dimensions.append({
            'width': img.width,
            'height': img.height,
            'aspect_ratio': img.width / img.height,
            'category': row['category']
        })
    except:
        continue

dim_df = pd.DataFrame(dimensions)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
# Width distribution
axes[0].hist(dim_df['width'], bins=30, edgecolor='black', alpha=0.7)
axes[0].set_title('Image Width Distribution', fontweight='bold')
axes[0].set_xlabel('Width (pixels)')
axes[0].set_ylabel('Frequency')
axes[0].axvline(dim_df['width'].mean(), color='red', linestyle='--', label=f'Mean: {dim_df["width"].mean():.0f}')
axes[0].legend()

# Height distribution
axes[1].hist(dim_df['height'], bins=30, edgecolor='black', alpha=0.7, color='orange')
axes[1].set_title('Image Height Distribution', fontweight='bold')
axes[1].set_xlabel('Height (pixels)')
axes[1].set_ylabel('Frequency')
axes[1].axvline(dim_df['height'].mean(), color='red', linestyle='--', label=f'Mean: {dim_df["height"].mean():.0f}')
axes[1].legend()

# Aspect ratio
axes[2].hist(dim_df['aspect_ratio'], bins=30, edgecolor='black', alpha=0.7, color='green')
axes[2].set_title('Aspect Ratio Distribution', fontweight='bold')
axes[2].set_xlabel('Aspect Ratio (W/H)')
axes[2].set_ylabel('Frequency')
axes[2].axvline(dim_df['aspect_ratio'].mean(), color='red', linestyle='--', label=f'Mean: {dim_df["aspect_ratio"].mean():.2f}')
axes[2].legend()

plt.tight_layout()
plt.savefig('../results/plots/image_dimensions.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n📊 Dimension Statistics:")
print(dim_df.describe())


In [ ]:
summary = {
    'Total Images': len(df),
    'Categories': df['category'].nunique(),
    'Avg Width': f"{dim_df['width'].mean():.0f} px",
    'Avg Height': f"{dim_df['height'].mean():.0f} px",
    'Most Common Category': category_counts.idxmax(),
    'Least Common Category': category_counts.idxmin(),
    'Class Imbalance Ratio': f"{category_counts.min() / category_counts.max():.2f}"
}
print("\n" + "="*50)
print("📋 DATASET SUMMARY")
print("="*50)
for key, value in summary.items():
    print(f"{key:.<30} {value}")
print("="*50)


In [ ]:
summary_df = pd.DataFrame([summary]).T
summary_df.columns = ['Value']
summary_df.to_csv('../results/reports/data_summary.csv')
print("\n✅ Summary saved to results/reports/data_summary.csv")